# Scenario & Workload Generator

Use this notebook to generate scenario and workload matrices based on template YAML files.

In [ ]:
# Install dependencies (PyYAML, ipywidgets, ipyfilechooser)
%pip install pyyaml ipywidgets ipyfilechooser

Note: you may need to restart the kernel to use updated packages.


In [23]:
from pathlib import Path
import sys
from IPython.display import display, Markdown
import ipywidgets as widgets
import subprocess
import shlex

NOTEBOOK_DIR = Path.cwd()
PROJECT_ROOT = NOTEBOOK_DIR.parent
SCENARIO_TEMPLATE = PROJECT_ROOT / 'data' / 'scenarios' / 'scenario_2.yml'
WORKLOAD_TEMPLATE = PROJECT_ROOT / 'data' / 'workloads' / 'workload_1.yml'
GENERATION_SCRIPT = PROJECT_ROOT / 'python' / 'generate_sim_data.py'
PYTHON_EXECUTABLE = sys.executable


## Permutation generation

In [ ]:
from ipyfilechooser import FileChooser

scenario_template = widgets.Text(value=str(SCENARIO_TEMPLATE), description="Template")
scenario_output = widgets.Text(value=str(PROJECT_ROOT / 'data' / 'scenarios' / 'generated'), description="Output dir")
scenario_base_name = widgets.Text(value="scenario", description="Base name")
scenario_sets = widgets.Textarea(value="system.cores=2,4\ntiming.context_switch_cost_us=25,50", description="--set entries", layout=widgets.Layout(width='600px', height='120px'))
scenario_run = widgets.Button(description="Generate", button_style="success", icon='play')
scenario_log = widgets.Output()

scenario_template.layout = widgets.Layout(width='100%')
scenario_output.layout = widgets.Layout(width='100%')
scenario_base_name.layout = widgets.Layout(width='50%')

initial_template = Path(scenario_template.value)
template_chooser = FileChooser(str(initial_template.parent))
template_chooser.title = 'Template file'
template_chooser.default_filename = initial_template.name

a = Path(scenario_output.value)
output_start = a if a.is_dir() else a.parent
output_chooser = FileChooser(str(output_start))
output_chooser.title = 'Output directory'
output_chooser.show_only_dirs = True


def handle_template_select(chooser):
    if chooser.selected:
        scenario_template.value = chooser.selected


def handle_output_select(chooser):
    selected = chooser.selected_path or chooser.selected
    if selected:
        scenario_output.value = selected


template_chooser.register_callback(handle_template_select)
output_chooser.register_callback(handle_output_select)


def run_scenario(_):
    scenario_log.clear_output()
    with scenario_log:
        cmd = [
            str(PYTHON_EXECUTABLE),
            str(GENERATION_SCRIPT),
            '--template', scenario_template.value,
            '--output-dir', scenario_output.value,
            '--base-name', scenario_base_name.value,
        ]
        for line in scenario_sets.value.splitlines():
            line = line.strip()
            if not line:
                continue
            cmd.extend(['--set', line])
        print('Running:', ' '.join(shlex.quote(part) for part in cmd))
        try:
            subprocess.run(cmd, check=True)
            print('Generation completed successfully.')
        except subprocess.CalledProcessError as exc:
            print(f'Generation failed: {exc}')


scenario_run.on_click(run_scenario)

ui = widgets.VBox([
    widgets.HBox([
        widgets.VBox([scenario_template, template_chooser], layout=widgets.Layout(width='50%')),
        widgets.VBox([scenario_output, output_chooser], layout=widgets.Layout(width='50%')),
    ]),
    scenario_base_name,
    scenario_sets,
    scenario_run,
    scenario_log
])

display(ui)
